# Downsampling and BIDS conversion

## Introduction
This section outlines how to prepare optically pumped magnetometer (OPM) data for further analysis. The first step is to downsample the data to an appropriate sampling frequency, which reduces file size and computational load while preserving the frequency range of interest. The data are then organised according to the MEG Brain Imaging Data Structure (MEG-BIDS) [1]. Adopting BIDS ensures that the necessary metadata are stored alongside the data and that a consistent, machine-readable file structure is used across datasets, which greatly simplifies sharing and downstream analysis. During the BIDS conversion, the trigger information is also decoded and labelled with meaningful event names.

By the end of this section you will be able to:
- downsample continuous MEG/OPM data with appropriate anti-aliasing filtering;
- decode multi-line trigger channels into a single event code and assign informative labels;
- write a valid, anonymised MEG-BIDS dataset using MNE-BIDS.

## Preparation
Import the required modules:

In [1]:
import os
import numpy as np
import pandas as pd
import mne
from mne_bids import (
    BIDSPath,
    make_dataset_description,
    print_dir_tree,
    read_raw_bids,
    write_meg_calibration,
    write_meg_crosstalk,
    write_raw_bids
)

## File overview

This section reads the raw FIF-files generated by the acquisition system. Because a single FIF-file cannot exceed 2 GB, the recording is split across several files; MNE-Python reassembles them automatically when the first file is read:
~~~
<ROOT>/20250410_110557_meg.fif
<ROOT>/20250410_110557_meg-1.fif
~~~
The downsampled data and the extracted events are then written locally:
~~~
<ROOT>/20250410_110557_rs_raw.fif
<ROOT>/20250410_110557_meg_eve.fif
~~~
Finally, the data are reorganised into a MEG-BIDS dataset with the following structure:
~~~
<ROOT>/Cerca_Spatt_BIDS/
├── dataset_description.json
├── participants.json
├── participants.tsv
└── sub-01/
    └── ses-01/
        ├── sub-01_ses-01_scans.tsv
        └── meg/
            ├── sub-01_ses-01_coordsystem.json
            ├── sub-01_ses-01_task-SpAtt_run-01_channels.tsv
            ├── sub-01_ses-01_task-SpAtt_run-01_events.json
            ├── sub-01_ses-01_task-SpAtt_run-01_events.tsv
            ├── sub-01_ses-01_task-SpAtt_run-01_meg.fif
            └── sub-01_ses-01_task-SpAtt_run-01_meg.json
~~~

## Downloading and storing the raw data locally

Visit [Zenodo](https://zenodo.org/records/17013192) and download the dataset.

## Importing the data

The OPM data are stored in FIF-format, a binary file structure with embedded labels. The first step is to define the path to the local data. **THIS IS USER-DEPENDENT**: edit `data_path` to point to the folder where you saved the downloaded data. As noted above, the acquisition system splits the recording across multiple files because a single FIF-file cannot exceed 2 GB; MNE-Python handles the continuation files (`*_meg-1.fif`, …) automatically, so you only need to point to the first file.

The resampled data will be written to a file with `rs_raw` in its name (`*rs_raw.fif`).

In [2]:
# The path below depends on where you stored the data locally. Edit it to match your system.
# data_path = '/path/to/your/data/CercaOxf/fif'
# data_path = 'E:/Data/Cerca'
data_path = '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA'
file_name = '20250410_110557_meg.fif'

raw_fname = os.path.join(data_path, file_name)

# Derived output file names
raw_resampled_fname = raw_fname.replace('meg.fif', 'rs_raw.fif')  # downsampled recording
event_fname = raw_fname.replace('.fif', '_eve.fif')               # extracted events
bids_folder = os.path.join(data_path, 'Cerca_Spatt_BIDS')         # BIDS output root

print('Raw file:      ', raw_fname)
print('Resampled file:', raw_resampled_fname)
print('Event file:    ', event_fname)

Raw file:       /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_meg.fif
Resampled file: /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_rs_raw.fif
Event file:     /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_meg_eve.fif


Now read the raw data:

In [3]:
raw = mne.io.read_raw_fif(raw_fname, preload=True)

Opening raw data file /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_meg.fif...
    Range : 0 ... 2449499 =      0.000 ...  1632.999 secs
Ready.
Opening raw data file /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_meg-1.fif...
    Range : 2449500 ... 3079615 =   1633.000 ...  2053.077 secs
Ready.
Reading 0 ... 3079615  =      0.000 ...  2053.077 secs...


To inspect the FIF-data write

In [4]:
print(raw.info)

<Info | 13 non-empty values
 bads: []
 ch_names: Trigger 1, Trigger 2, Trigger 3, Trigger 4, Trigger 5, Trigger ...
 chs: 27 Stimulus, 192 Magnetometers
 custom_ref_applied: False
 dev_head_t: MEG device -> head transform
 device_info: 2 items (dict)
 dig: 15603 items (3 Cardinal, 15600 Extra)
 file_id: 4 items (dict)
 highpass: 0.0 Hz
 line_freq: 0.0
 lowpass: 750.0 Hz
 meas_date: unspecified
 meas_id: 4 items (dict)
 nchan: 219
 projs: []
 sfreq: 1500.0 Hz
>


## Down-sampling the raw data

The next step is to downsample the data to an appropriate sampling frequency. Downsampling reduces file size and speeds up subsequent processing. To avoid aliasing, the data must first be low-pass filtered below the new Nyquist frequency (half the target sampling rate); in practice a cutoff of about 1/4 to 1/3 of the target sampling rate is used so that the filter's soft roll-off is fully below Nyquist [2].

Here the data are resampled from 1500 Hz to 750 Hz. Before resampling, a finite impulse response (FIR) low-pass filter is applied at 750 Hz / 3 = 250 Hz, comfortably below the new Nyquist frequency of 375 Hz. This retains neuronal activity up to 250 Hz, which is more than sufficient for most applications. At the same time a 0.1 Hz high-pass filter is applied to remove slow drifts.

> **Note on event timing.** Resampling stim/trigger channels distorts the sharp edges that encode events (MNE-Python warns about this). We therefore decode the events from the trigger lines on the *original* 1500 Hz data first, and pass them to `resample(..., events=events)` so their onsets are realigned jointly with the data. The trigger channels themselves are still resampled, but their timing is no longer relied upon.

### Decoding the triggers before resampling

Because resampling the trigger channels is unreliable, we decode the events *before* downsampling. The event (trial) information is encoded across several digital lines — here Trigger 5 to Trigger 11 (seven channels), though this is highly system-dependent. Each line is a separate digital signal driven by the stimulus computer and is either off (0) or on (1) at any moment.

To recover the events, the seven binary lines are combined into a single integer code, which we write onto the `Trigger 1` channel. Each line `n` (0-indexed) is given a binary weight of $2^{\,n+1}$, so Trigger 5 contributes 2, Trigger 6 contributes 4, Trigger 7 contributes 8, and so on. Because the weights are distinct powers of two, the combined value uniquely identifies *which* lines were high: a single line firing gives a pure power of two (e.g. 2, 4, 8), whereas two lines firing together give their sum (e.g. 2 + 4 = 6). This weighting scheme is itself system- and experiment-dependent. The resulting event array is then passed to `resample` so that the onsets are rescaled jointly with the data.

In [5]:
raw.load_data()

# The seven digital trigger lines used in this experiment
trigger_chs = ['Trigger 5', 'Trigger 6', 'Trigger 7', 'Trigger 8',
               'Trigger 9', 'Trigger 10', 'Trigger 11']

# Combine the binary lines into a single integer code using weights 2^(n+1)
trigger_data = raw.copy().pick(trigger_chs).get_data()
combined_trigger = np.zeros(trigger_data.shape[1], dtype=int)
for index_ch, ch_data in enumerate(trigger_data):
    combined_trigger += ch_data.astype(int) * (2 ** (index_ch + 1))

# Write the combined code onto the 'Trigger 1' channel and detect the events
# on the ORIGINAL 1500 Hz data (before resampling), so their timing is reliable
trigger1_idx = raw.ch_names.index('Trigger 1')
raw._data[trigger1_idx, :] = combined_trigger
events = mne.find_events(raw, stim_channel='Trigger 1', min_duration=0.003)

print(events[:10])

Finding events on: Trigger 1
1365 events found on stim channel Trigger 1
Event IDs: [ 2  4  8 10 12 32 34 36 64]
[[175812      0     10]
 [180735      0      4]
 [182540      0      8]
 [183349      0     36]
 [183927      0     64]
 [188160      0      4]
 [189965      0      8]
 [190824      0     36]
 [191451      0     64]
 [195760      0      4]]


Each row of `events` has three columns: the sample index, the (usually unused) preceding value, and the event code. For example, the second row `[90367  0  4]` shows that at sample 90367 there is an event with code 4, which we label `cue_Left` in the BIDS section below. These sample indices are at the original 1500 Hz; `resample` rescales them to the 750 Hz grid when we downsample. The numeric codes are given informative labels later, in the BIDS conversion.

In [6]:
desired_sfreq = 750                       # target sampling frequency (Hz)
current_sfreq = raw.info['sfreq']         # original sampling frequency (Hz)

lowpass_freq = desired_sfreq / 3.0        # anti-aliasing cutoff, well below Nyquist (375 Hz)
highpass_freq = 0.1                       # remove slow drifts

print(f'Resampling from {current_sfreq:.0f} Hz to {desired_sfreq} Hz '
      f'(band-pass {highpass_freq}-{lowpass_freq:.0f} Hz applied first)')

# Filter on a copy so the original raw object is left untouched
raw_resampled = raw.copy().filter(l_freq=highpass_freq, h_freq=lowpass_freq)

# Resample the data and realign the events jointly. Passing events= avoids relying on
# the (unreliable) resampled trigger channels; the returned events are on the 750 Hz grid.
raw_resampled, events = raw_resampled.resample(sfreq=desired_sfreq, events=events)

Resampling from 1500 Hz to 750 Hz (band-pass 0.1-250 Hz applied first)
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 2.5e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 250.00 Hz
- Upper transition bandwidth: 62.50 Hz (-6 dB cutoff frequency: 281.25 Hz)
- Filter length: 49501 samples (33.001 s)



**Question 1** Why is a highpass filter of 0.1 Hz applied, and what would happen to slow evoked responses if this cutoff were set too high (e.g. 1 Hz)?


Subsequently, the down-sampled data are stored locally: 

In [7]:
raw_resampled.save(raw_resampled_fname, overwrite=True)

# Save the (resampled) events for the BIDS conversion
mne.write_events(event_fname, events, overwrite=True)

Writing /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_rs_raw.fif
Closing /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_rs_raw.fif
[done]


In [8]:
print(raw_resampled.info)

<Info | 13 non-empty values
 bads: []
 ch_names: Trigger 1, Trigger 2, Trigger 3, Trigger 4, Trigger 5, Trigger ...
 chs: 27 Stimulus, 192 Magnetometers
 custom_ref_applied: False
 dev_head_t: MEG device -> head transform
 device_info: 2 items (dict)
 dig: 15603 items (3 Cardinal, 15600 Extra)
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 line_freq: 0.0
 lowpass: 250.0 Hz
 meas_date: unspecified
 meas_id: 4 items (dict)
 nchan: 219
 projs: []
 sfreq: 750.0 Hz
>


The rest of the tutorial will be based on the resampled data. The original FIF-data with the 1500 Hz sample rate can therefore be archived.

## Converting to MEG BIDS format

The next step is to organize the resampled FIF-data according to the BIDS convention. This includes identifying and naming the trigger information. Start by reading the resampled data:

In [9]:
del raw, raw_resampled
raw = mne.io.read_raw(raw_resampled_fname)

Opening raw data file /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/20250410_110557_rs_raw.fif...
    Range : 0 ... 1539807 =      0.000 ...  2053.076 secs
Ready.


### Assigning labels to the trigger codes

The events were already decoded from the trigger lines and rescaled to 750 Hz in the previous section, and saved to the event file. What remains before writing the BIDS dataset is to give each numeric code an informative label.

In [10]:
# Map each combined trigger code to an informative label.
# The comments show which digital lines are high (T5=2, T6=4, T7=8, T8=16, T9=32, T10=64).
event_dict = {
    'off':         0,   # no line high
    'cue_Right':   2,   # T5
    'cue_Left':    4,   # T6
    'trial_Start': 6,   # T5 + T6
    'stimOnset':   8,   # T7
    'blkStart':   10,   # T7 + T5
    'blkEnd':     12,   # T7 + T6
    'expEnd':     18,   # T8 + T5
    'abort':      20,   # T8 + T6
    'catchOnset': 32,   # T9
    'dotOnRight': 34,   # T9 + T5
    'dotOnLeft':  36,   # T9 + T6
    'resp':       64,   # T10
}

In this example, trigger value 6 denotes the onset of a trial and is therefore labelled `trial_Start`. Likewise `cue_Right` (code 2) and `cue_Left` (code 4) denote the cues instructing the participant to attend right or left, respectively. Choosing informative labels here pays off later, as they propagate into the BIDS `events.tsv` files and every downstream analysis. Labels must be chosen to match the specific study design.

**Question 2** Some codes (e.g. 6, 10, 12, 34, 36) are sums of two powers of two, while others (2, 4, 8, 32, 64) are single powers of two. What does this tell you about how many trigger lines were high at the same time for each event, and why is a distinct-power-of-two weighting essential for the decoding to be unambiguous?

### Organizing and storing the data according to BIDS

For the BIDS conversion, several parameters must be defined according to the subject and session number. 

In [11]:
raw.info["line_freq"] = 50  # UK/EU mains frequency; use 60 for the US and other 60 Hz regions
raw.set_annotations(None)
subject = '01'
session = '01'
task = 'SpAtt'
run = '01'

bids_path = BIDSPath(
    subject=subject, 
    session=session, 
    task=task, 
    run=run, 
    datatype="meg", 
    root=bids_folder
)
write_raw_bids(
    raw=raw,
    bids_path=bids_path,
    events=event_fname,
    event_id=event_dict,
    overwrite=True,
    allow_preload=True, 
    format='FIF',
    # daysback shifts all dates in the file backward by a fixed random offset for anonymization;
    # keep_his/keep_source control whether identifying history/source info is retained
    anonymize={'daysback': 50000, 'keep_his': False, 'keep_source': False}
)

Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/README'...
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/participants.tsv'...
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/participants.json'...
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/sub-01/ses-01/meg/sub-01_ses-01_coordsystem.json'...
Used Annotations descriptions: [np.str_('blkEnd'), np.str_('blkStart'), np.str_('catchOnset'), np.str_('cue_Left'), np.str_('cue_Right'), np.str_('dotOnLeft'), np.str_('dotOnRight'), np.str_('resp'), np.str_('stimOnset')]
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/sub-01/ses-01/meg/sub-01_ses-01_task-SpAtt_run-01_events.tsv'...
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/sub-01/ses-01/meg/sub-01_ses-01_task-SpAtt_run-01_events.json'...
Writing '/Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS/dataset_description.json'...


BIDSPath(
root: /Users/o.jensen@bham.ac.uk/Data/CercaOxf/subjA/Cerca_Spatt_BIDS
datatype: meg
basename: sub-01_ses-01_task-SpAtt_run-01_meg.fif)

To explore the data organization according to BIDS inspect the folders: 

In [12]:
print_dir_tree(bids_folder)

|Cerca_Spatt_BIDS/
|--- README
|--- dataset_description.json
|--- participants.json
|--- participants.tsv
|--- sub-01/
|------ ses-01/
|--------- sub-01_ses-01_scans.tsv
|--------- meg/
|------------ sub-01_ses-01_coordsystem.json
|------------ sub-01_ses-01_task-SpAtt_run-01_channels.tsv
|------------ sub-01_ses-01_task-SpAtt_run-01_events.json
|------------ sub-01_ses-01_task-SpAtt_run-01_events.tsv
|------------ sub-01_ses-01_task-SpAtt_run-01_meg.fif
|------------ sub-01_ses-01_task-SpAtt_run-01_meg.json


Some of these files are useful for inspecting the data. For instance, `sub-01_ses-01_task-SpAtt_run-01_events.tsv` is a plain-text (tab-separated) file listing the events and their labels, and can be opened in any text editor or spreadsheet program.

**Question 3** Inspect the file `sub-01_ses-01_task-SpAtt_run-01_events.tsv` and report the time (in seconds) and sample point of the first occurrence of the `cue_Left` and `cue_Right` triggers, respectively.

## Preregistration and publication

A preregistration should specify the preprocessing choices made in this section in enough detail that another researcher could reproduce them exactly. For the downsampling and BIDS-conversion stage, we recommend committing to the following in advance:

**Downsampling and filtering**
- Original and target sampling frequencies (here 1500 Hz → 750 Hz).
- Anti-aliasing filter: type (FIR), cutoff (250 Hz), and the fact that it is applied *before* resampling. State that the cutoff is set below the new Nyquist frequency (375 Hz).
- High-pass filter cutoff (0.1 Hz) and its purpose (removing slow drifts), noting that this can attenuate very slow evoked components.
- Software and version used (e.g. `MNE-Python 1.x`), since default filter parameters can change between releases.

**Event/trigger definitions**
- Which trigger lines are used and the weighting scheme that combines them into event codes.
- The full mapping from codes to condition labels (the `event_dict` above), ideally as a table, together with the intended experimental meaning of each label.
- The event(s) that will serve as the epoching time-lock point in later analyses (e.g. `cue_Left` / `cue_Right`).

**Data organisation and sharing**
- That data will be organised in MEG-BIDS [1] and, if applicable, deposited in a public repository (e.g. Zenodo/OpenNeuro).
- The anonymisation strategy (here, date shifting via `anonymize={'daysback': ...}` and removal of identifying history).

Distinguish clearly between choices fixed in advance (confirmatory) and any that may be adjusted after inspecting the data (exploratory).

**Example preregistration / methods text**

> "OPM-MEG data were acquired at 1500 Hz. Prior to analysis the continuous data were band-pass filtered between 0.1 Hz and 250 Hz using a zero-phase FIR filter and downsampled to 750 Hz. The 250 Hz low-pass cutoff (one third of the target sampling rate) served as an anti-aliasing filter below the 375 Hz Nyquist frequency. Stimulus triggers were decoded from seven digital lines into condition-specific event codes and labelled according to the experimental design. All processing was performed in MNE-Python (version X.X) and MNE-BIDS (version X.X), and the resulting data were organised according to the MEG-BIDS specification [1] and anonymised by date-shifting."

## References

[1] Niso G, Gorgolewski KJ, Bock E, Brooks TL, Flandin G, Gramfort A, Henson RN, Jas M, Litvak V, Moreau JT, Oostenveld R, Schoffelen JM, Tadel F, Wexler J, Baillet S. MEG-BIDS, the brain imaging data structure extended to magnetoencephalography. *Scientific Data* 5, 180110 (2018). [doi:10.1038/sdata.2018.110](https://doi.org/10.1038/sdata.2018.110).

[2] Smith SW. *The Scientist and Engineer's Guide to Digital Signal Processing*. California Technical Publishing, 1998. [PDF](https://www.dspguide.com/pdfbook.htm).
